# Policy Inference Latency Benchmark (Pipelined)

**Pipelined Action Chunking** 방식의 실시간성을 평가합니다.

## 기존 방식 (Blocking)
```
Step 0:  [forward pass ~23ms] → action[0] → env.step()
Step 1:  action[1] (cached)   → env.step()
...
Step 9:  action[9] (cached)   → env.step()
Step 10: [forward pass ~23ms] → action[0] → env.step()  ← 여기서 blocking
```
→ forward pass가 **50ms (1 dt)** 이내에 완료되어야 함

## Pipelined 방식 (이 노트북)
```
Step 0:  action[0] (이전에 계산됨) → env.step() + [forward 시작 (다음 chunk)]
Step 1:  action[1] (cached)        → env.step() + [forward 계속...]
...
Step 9:  action[9] (cached)        → env.step() + [forward 완료]
Step 10: action[0] (새 chunk 준비됨) → env.step() + [forward 시작 (다다음 chunk)]
```
→ forward pass가 **500ms (dt × chunk_size)** 이내에 완료되면 됨
→ 현재 chunk를 소비하는 동안 다음 chunk를 비동기로 계산

In [1]:
import os
import sys
import time
import threading
import numpy as np
import torch

# GPU 설정
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

sys.path.insert(0, os.path.dirname(os.path.abspath('')))
sys.path.insert(0, os.path.abspath('.'))
if '/Data/pilab/LIBERO' not in sys.path:
    sys.path.insert(0, '/Data/pilab/LIBERO')

from configs.config import create_libero_object_config
from configs.factory import create_model, create_trainer

In [2]:
# 설정
BENCHMARK_TYPE = 'libero_object'
CHECKPOINT_PATH = 'runs/libero_object/2026-03-12/05-41-43/final_model.pth'
DEVICE = 'cuda'
TASK_ID = 0

In [3]:
# 모델 로드
cfg = create_libero_object_config()
model = create_model(cfg)
trainer = create_trainer(cfg)
model.set_scaler(trainer.scaler)

state_dict = torch.load(CHECKPOINT_PATH, weights_only=True)
model.load_state_dict(state_dict)
model = model.to(DEVICE)
model.eval()
print(f'Model loaded from {CHECKPOINT_PATH}')

/Data/pilab/anaconda3/envs/mamba3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Data/pilab/anaconda3/envs/mamba3/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Data/pilab/anaconda3/envs/mamba3/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Model loaded from runs/libero_object/2026-03-12/05-41-43/final_model.pth


In [4]:
# 환경 설정
from libero.libero import benchmark
from libero.libero.envs import OffScreenRenderEnv

benchmark_suite = benchmark.get_benchmark_dict()[BENCHMARK_TYPE]()
task_bddl_file = benchmark_suite.get_task_bddl_file_path(TASK_ID)
file_name = os.path.basename(task_bddl_file).split('.')[0]
task_emb = trainer.trainset.tasks[file_name].to(DEVICE).unsqueeze(0)
init_states = benchmark_suite.get_task_init_states(TASK_ID)

env = OffScreenRenderEnv(
    bddl_file_name=task_bddl_file,
    camera_heights=128,
    camera_widths=128
)
env.seed(0)
env.reset()
obs = env.set_init_state(init_state=init_states[0])

dummy = np.zeros(7)
dummy[-1] = -1.0
for _ in range(5):
    obs, _, _, _ = env.step(dummy)

model.reset()
ACTION_SEQ_LEN = model.action_seq_len
print(f'Task: {file_name}')
print(f'Action sequence length: {ACTION_SEQ_LEN}')

[robosuite WARNING] No private macro file found! (__init__.py:7)
[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)
[robosuite WARNING] To setup, run: python /Data/pilab/anaconda3/envs/mamba3/lib/python3.10/site-packages/robosuite/scripts/setup_macros.py (__init__.py:9)
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Task: pick_up_the_alphabet_soup_and_place_it_in_the_basket
Action sequence length: 10


In [5]:
# Pipelined 실행 시뮬레이션
#
# 구조:
#   chunk N 소비 중 (step 0~9) → chunk N+1을 비동기 forward
#   chunk N+1은 step 0 시점의 obs로 forward (가장 최근 관측)
#   chunk N의 마지막 action(step 9) 이전에 forward 완료되어야 함
#
# 측정:
#   - forward_latency: 각 chunk의 forward pass 시간
#   - chunk_budget: chunk 소비에 걸린 wall clock 시간 (= forward에 사용 가능한 시간)
#   - deadline_met: forward가 chunk 소비 내에 완료되었는지

MAX_STEPS = 600

# GPU warmup
model.reset()
with torch.no_grad():
    for _ in range(3):
        dummy_obs = {
            'agentview_image': torch.randn(1, 1, 3, 128, 128).to(DEVICE),
            'eye_in_hand_image': torch.randn(1, 1, 3, 128, 128).to(DEVICE),
            'lang_emb': task_emb,
            'robot_states': torch.randn(1, 1, 9).to(DEVICE),
        }
        _ = model.predict(dummy_obs)
    model.reset()
    torch.cuda.synchronize()


def make_obs_dict(obs):
    """numpy obs → GPU tensor obs_dict"""
    agentview_rgb = torch.from_numpy(obs['agentview_image']).to(DEVICE).float().permute(2, 0, 1).unsqueeze(0).unsqueeze(0) / 255.
    eye_in_hand_rgb = torch.from_numpy(obs['robot0_eye_in_hand_image']).to(DEVICE).float().permute(2, 0, 1).unsqueeze(0).unsqueeze(0) / 255.
    joint_state = obs['robot0_joint_pos']
    gripper_state = obs['robot0_gripper_qpos']
    robot_states = torch.from_numpy(np.concatenate([joint_state, gripper_state], axis=-1)).to(DEVICE).float().unsqueeze(0).unsqueeze(0)
    return {
        'agentview_image': agentview_rgb,
        'eye_in_hand_image': eye_in_hand_rgb,
        'lang_emb': task_emb,
        'robot_states': robot_states,
    }


def run_forward(model, obs_dict):
    """model forward pass (action chunk 생성)"""
    with torch.no_grad():
        # model.predict()의 forward 부분만 직접 호출
        # obs_seq 처리
        if not model.obs_seq:
            for key in obs_dict.keys():
                from collections import deque
                model.obs_seq[key] = deque(maxlen=model.perception_seq_len)

        for key in obs_dict.keys():
            model.obs_seq[key].append(obs_dict[key])
            if key == 'lang' or key == 'lang_emb':
                continue
            obs_dict[key] = torch.concat(list(model.obs_seq[key]), dim=1)
            if obs_dict[key].shape[1] < model.perception_seq_len:
                import einops
                pad = einops.repeat(
                    obs_dict[key][:, 0],
                    'b ... -> b t ...',
                    t=model.perception_seq_len - obs_dict[key].shape[1],
                )
                obs_dict[key] = torch.cat([pad, obs_dict[key]], dim=1)

        model.eval()
        pred_action_seq = model(obs_dict)[:, :model.action_seq_len]
        pred_action_seq = model.scaler.inverse_scale_output(pred_action_seq)
        torch.cuda.synchronize()
    return pred_action_seq


# ============================================================
# 측정 시작
# ============================================================
print('Running pipelined episode...')

forward_times = []           # 각 chunk의 forward 시간 (ms)
chunk_exec_times = []        # 각 chunk의 실행 시간 = forward 가용 시간 (ms)
preprocess_times = []        # obs preprocessing 시간 (ms)
env_step_times = []          # env.step 시간 (ms)
total_step_times = []        # 전체 step 시간 (ms)
deadline_results = []        # 각 chunk: forward가 시간 내 완료되었는가

model.reset()
step = 0
r = 0

# 첫 번째 chunk: blocking forward (아직 소비할 캐시 없음)
t_pre = time.perf_counter()
obs_dict = make_obs_dict(obs)
torch.cuda.synchronize()
t_pre_end = time.perf_counter()

t_fwd = time.perf_counter()
current_chunk = run_forward(model, obs_dict)
t_fwd_end = time.perf_counter()
first_forward_ms = (t_fwd_end - t_fwd) * 1000
print(f'  First chunk forward: {first_forward_ms:.1f} ms (blocking, no pipeline)')

while step < MAX_STEPS:
    chunk_start_time = time.perf_counter()
    next_chunk = None
    next_forward_time = None

    # 현재 chunk의 action을 순차 소비
    for action_idx in range(ACTION_SEQ_LEN):
        t_step_start = time.perf_counter()

        # 현재 action 실행
        action = current_chunk[0, action_idx].cpu().numpy()

        t_env = time.perf_counter()
        obs, r, done, _ = env.step(action)
        t_env_end = time.perf_counter()
        env_step_times.append((t_env_end - t_env) * 1000)

        step += 1
        total_step_times.append((t_env_end - t_step_start) * 1000)

        # 첫 action 실행 직후 다음 chunk forward 시작 (pipelined)
        if action_idx == 0 and next_chunk is None:
            t_pre = time.perf_counter()
            next_obs_dict = make_obs_dict(obs)
            torch.cuda.synchronize()
            t_pre_end = time.perf_counter()
            preprocess_times.append((t_pre_end - t_pre) * 1000)

            t_fwd = time.perf_counter()
            next_chunk = run_forward(model, next_obs_dict)
            t_fwd_end = time.perf_counter()
            next_forward_time = (t_fwd_end - t_fwd) * 1000
            forward_times.append(next_forward_time)

        if r == 1:
            print(f'SUCCESS at step {step}')
            break
        if step >= MAX_STEPS:
            break

    chunk_end_time = time.perf_counter()
    chunk_wall_ms = (chunk_end_time - chunk_start_time) * 1000
    chunk_exec_times.append(chunk_wall_ms)

    # Deadline 판정: forward가 chunk 소비 시간 내에 완료되었는가
    if next_forward_time is not None:
        deadline_results.append({
            'forward_ms': next_forward_time,
            'budget_ms': chunk_wall_ms,
            'met': next_forward_time <= chunk_wall_ms,
        })

    if r == 1 or step >= MAX_STEPS:
        break

    # 다음 chunk로 전환
    current_chunk = next_chunk

if r != 1:
    print(f'FAILED after {MAX_STEPS} steps')

env.close()
print(f'Total steps: {step}')
print(f'Total chunks: {len(forward_times)}')

Running pipelined episode...
  First chunk forward: 22.5 ms (blocking, no pipeline)
FAILED after 600 steps
Total steps: 600
Total chunks: 60


In [6]:
# 결과 출력
CONTROL_FREQ = 20  # LIBERO control frequency (Hz)
SIM_DT = 1000.0 / CONTROL_FREQ  # 50ms
CHUNK_BUDGET = SIM_DT * ACTION_SEQ_LEN  # 500ms

def print_stats(name, times_ms):
    arr = np.array(times_ms)
    print(f'\n  --- {name} ---')
    print(f'    Mean:  {arr.mean():.3f} ms')
    print(f'    Std:   {arr.std():.3f} ms')
    print(f'    Min:   {arr.min():.3f} ms')
    print(f'    Max:   {arr.max():.3f} ms')
    print(f'    P50:   {np.percentile(arr, 50):.3f} ms')
    print(f'    P95:   {np.percentile(arr, 95):.3f} ms')
    print(f'    P99:   {np.percentile(arr, 99):.3f} ms')

# ============================================================
# 1. 에피소드 결과
# ============================================================
print('=' * 70)
print('  1. Episode Result')
print('=' * 70)
success = (r == 1)
print(f'  Task:    {file_name}')
print(f'  Result:  {"SUCCESS" if success else "FAILED"}')
print(f'  Steps:   {step} / {MAX_STEPS}')
print(f'  Chunks:  {len(forward_times)}')
if success:
    print(f'  Episode time (wall clock): {sum(total_step_times)/1000:.2f} s')

# ============================================================
# 2. 방식 비교
# ============================================================
print('\n' + '=' * 70)
print('  2. Blocking vs Pipelined 비교')
print('=' * 70)
print(f'                        Blocking         Pipelined')
print(f'  Forward deadline:     {SIM_DT:.0f} ms (1 dt)      {CHUNK_BUDGET:.0f} ms ({ACTION_SEQ_LEN} dt)')
print(f'  Forward timing:       매 chunk 시작 시   이전 chunk 소비 중')
print(f'  Obs 시점:             현재 step         이전 chunk의 step 1')
print(f'  첫 chunk:             blocking          blocking (동일)')

# ============================================================
# 3. LIBERO 시뮬레이터 사양
# ============================================================
print('\n' + '=' * 70)
print('  3. LIBERO Simulator Specification')
print('=' * 70)
print(f'  Control frequency:  {CONTROL_FREQ} Hz')
print(f'  Sim dt:             {SIM_DT:.1f} ms')
print(f'  Chunk budget:       {CHUNK_BUDGET:.1f} ms (dt x {ACTION_SEQ_LEN})')
print(f'  Action dim:         7 (6 DoF + gripper)')
print(f'  Image resolution:   128 x 128')

# ============================================================
# 4. Latency 상세
# ============================================================
print('\n' + '=' * 70)
print('  4. Latency Breakdown')
print('=' * 70)

print_stats('Forward Pass (per chunk)', forward_times)
print_stats('Chunk Execution (wall clock)', chunk_exec_times)
if preprocess_times:
    print_stats('Obs Preprocessing', preprocess_times)
print_stats('Env Step', env_step_times)
print_stats('Total per Step', total_step_times)

# ============================================================
# 5. 실시간성 판정 (Pipelined)
# ============================================================
print('\n' + '=' * 70)
print('  5. Real-time Feasibility (Pipelined)')
print('=' * 70)

fwd_arr = np.array(forward_times)
budget_arr = np.array([d['budget_ms'] for d in deadline_results])
met_arr = np.array([d['met'] for d in deadline_results])

print(f'  Chunk budget (이론):        {CHUNK_BUDGET:.1f} ms ({ACTION_SEQ_LEN} x {SIM_DT:.0f} ms)')
print(f'  Chunk budget (실측 평균):   {budget_arr.mean():.1f} ms')
print()
print(f'  Forward pass:')
print(f'    Mean:                     {fwd_arr.mean():.3f} ms')
print(f'    Max:                      {fwd_arr.max():.3f} ms')
print(f'    P95:                      {np.percentile(fwd_arr, 95):.3f} ms')
print()
print(f'  Deadline 충족:')
print(f'    충족 횟수:                {met_arr.sum():.0f} / {len(met_arr)}')
print(f'    충족률:                   {met_arr.mean() * 100:.1f}%')
if not met_arr.all():
    miss_indices = np.where(~met_arr)[0]
    for idx in miss_indices:
        d = deadline_results[idx]
        print(f'    MISS chunk {idx}: forward {d["forward_ms"]:.1f} ms > budget {d["budget_ms"]:.1f} ms')
print()

# 여유율
margin_mean = (budget_arr.mean() - fwd_arr.mean()) / budget_arr.mean() * 100
margin_worst = (budget_arr.min() - fwd_arr.max()) / budget_arr.min() * 100

print(f'  여유율 (평균):              {margin_mean:.1f}%')
print(f'  여유율 (worst case):        {margin_worst:.1f}%')
print()

# 이론적 한계: forward가 최대 몇 ms까지 허용 가능한가
print(f'  이론적 forward 허용 한계:   {CHUNK_BUDGET:.0f} ms')
print(f'  현재 forward mean:          {fwd_arr.mean():.1f} ms')
print(f'  → 모델을 {CHUNK_BUDGET / fwd_arr.mean():.1f}x 키워도 실시간 유지 가능')

# ============================================================
# 6. Blocking vs Pipelined 최종 비교
# ============================================================
print('\n' + '=' * 70)
print('  6. Summary: Blocking vs Pipelined')
print('=' * 70)
fwd_mean = fwd_arr.mean()
fwd_max = fwd_arr.max()

blocking_ok_mean = fwd_mean < SIM_DT
blocking_ok_max = fwd_max < SIM_DT
pipelined_ok_mean = fwd_mean < CHUNK_BUDGET
pipelined_ok_max = fwd_max < CHUNK_BUDGET

print(f'                        Blocking ({SIM_DT:.0f}ms)    Pipelined ({CHUNK_BUDGET:.0f}ms)')
print(f'  Forward mean:         {"✓ OK" if blocking_ok_mean else "✗ FAIL":14s}  {"✓ OK" if pipelined_ok_mean else "✗ FAIL"}')
print(f'    ({fwd_mean:.1f} ms)          margin {SIM_DT - fwd_mean:+.1f} ms      margin {CHUNK_BUDGET - fwd_mean:+.1f} ms')
print(f'  Forward max:          {"✓ OK" if blocking_ok_max else "✗ FAIL":14s}  {"✓ OK" if pipelined_ok_max else "✗ FAIL"}')
print(f'    ({fwd_max:.1f} ms)          margin {SIM_DT - fwd_max:+.1f} ms      margin {CHUNK_BUDGET - fwd_max:+.1f} ms')
print()
print(f'  Pipelined 장점: forward budget이 {CHUNK_BUDGET/SIM_DT:.0f}x 증가 ({SIM_DT:.0f}ms → {CHUNK_BUDGET:.0f}ms)')
print(f'  Pipelined 단점: obs가 최대 {(ACTION_SEQ_LEN-1) * SIM_DT:.0f}ms 지연 (chunk 시작 시점 기준)')
print('=' * 70)

  1. Episode Result
  Task:    pick_up_the_alphabet_soup_and_place_it_in_the_basket
  Result:  FAILED
  Steps:   600 / 600
  Chunks:  60

  2. Blocking vs Pipelined 비교
                        Blocking         Pipelined
  Forward deadline:     50 ms (1 dt)      500 ms (10 dt)
  Forward timing:       매 chunk 시작 시   이전 chunk 소비 중
  Obs 시점:             현재 step         이전 chunk의 step 1
  첫 chunk:             blocking          blocking (동일)

  3. LIBERO Simulator Specification
  Control frequency:  20 Hz
  Sim dt:             50.0 ms
  Chunk budget:       500.0 ms (dt x 10)
  Action dim:         7 (6 DoF + gripper)
  Image resolution:   128 x 128

  4. Latency Breakdown

  --- Forward Pass (per chunk) ---
    Mean:  23.686 ms
    Std:   1.980 ms
    Min:   21.309 ms
    Max:   31.307 ms
    P50:   23.507 ms
    P95:   27.417 ms
    P99:   29.561 ms

  --- Chunk Execution (wall clock) ---
    Mean:  166.128 ms
    Std:   12.795 ms
    Min:   140.135 ms
    Max:   190.232 ms
    P50:   170.635